# 02 · Train one variant

| Setting | Value |
|---|---|
| Accelerator | **GPU L4** (bfloat16, preferred) or **GPU T4 x2** (float16) — set `HW` |
| Internet | **ON** (pip install + model download). OFF works if the model is attached and `model.source=local` |
| Attach | code (`scopesum-code`) unless `REPO_URL`; **data** from 01 (e.g. `scopesum-data`); to resume: the previous session's output (e.g. `scopesum-run-C`) |
| Expected runtime | depends on tokens/sec printed by 00; 10M tokens at ~1.5k tok/s ≈ 2 h fwd+bwd per variant, plus overhead. Sessions stop cleanly after `train.max_hours` |

**Resume across sessions:** after a session ends, *Save Version* → create a dataset from this notebook's output (`/kaggle/working/scopesum_runs`) → attach it to the next session and set `RESUME_FROM` to its path. The newest `ckpt-*` inside is used (LoRA, `<sum>` embedding, optimizer, scheduler, RNG, data position).

In [ ]:
VARIANT = "C"         # A | B | C | D | E | F | G  <- the only thing to change per run
HW = "t4"             # "t4" (float16; Kaggle GPU T4 x2) or "l4" (bfloat16)
DATA_DIR = "/kaggle/input/scopesum-data/scopesum_data"   # output of 01 (attached dataset); VERIFY path
RESUME_FROM = None    # e.g. "/kaggle/input/scopesum-run-c/scopesum_runs/variant_C"
MAX_STEPS = 130       # 130 steps x 16 windows x ~1436 tokens = ~3M tokens per variant (None = all 10M)
WARMUP_STEPS = 13     # 10% of MAX_STEPS; None = config default (50)

# --- get the code: git clone (needs Internet) OR an attached Kaggle dataset ---
REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"
import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

In [ ]:
!pip uninstall -y -q torchao  # Kaggle ships torchao 0.10; peft>=0.21 refuses it and we do not use torchao
!pip install -q -r requirements.txt

In [ ]:
import json
from scopesum.config import load_config
from scopesum.data import load_stats

if not os.path.exists(os.path.join(DATA_DIR, "stats.json")):
    # only a stats.json that sits next to the prepared windows (the repo also has a copy in results/)
    found = [p for p in glob.glob("/kaggle/input/**/stats.json", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "windows_train"))]
    print("stats.json not at DATA_DIR; found:", found)
    DATA_DIR = os.path.dirname(found[0])
overrides = {"data": {"out_dir": DATA_DIR}, "train": {"resume_from": RESUME_FROM, "max_steps": MAX_STEPS}}
if WARMUP_STEPS is not None:
    overrides["train"]["warmup_steps"] = WARMUP_STEPS
cfg = load_config("configs/base.yaml", f"configs/variant_{VARIANT}.yaml", f"configs/hw_{HW}.yaml", overrides=overrides)
stats = load_stats(DATA_DIR)
print("data stats:", json.dumps({k: stats[k] for k in ["n_examples", "raw_tokens_total", "chunk_size", "avg_scope_len", "avg_scope_sums_per_example", "avg_fixed_sums_per_example"]}, indent=2))
print(cfg.variant, cfg.train, sep="\n")

In [ ]:
from scopesum.train import train
final_dir = train(cfg)
print("final / latest checkpoint:", final_dir)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
log = pd.read_csv(os.path.join(cfg.train.out_dir, f"variant_{VARIANT}", "train_log.csv"))
print(log.tail())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log.step, log.loss); ax[0].set_title(f"variant {VARIANT}: train loss"); ax[0].set_xlabel("step")
ax[1].plot(log.step, log.tokens_per_sec); ax[1].set_title("tokens / sec"); ax[1].set_xlabel("step")
plt.tight_layout(); plt.show()
!ls -la {cfg.train.out_dir}/variant_{VARIANT}